# OpenINTEL Split Template
## Analysis book

In [1]:
# Import oi_pyspark with the oi_start and oi_stop functions
%run ./oi_pyspark_session.ipynb

SparkConf created


In [2]:
# Start the actual Spark session
oi_start()

Started SparkSession


In [17]:
# Clear spark catalog
oi_clearcache()

In [18]:
# Stop spark session
oi_stop()

Cleared cache and stopped spark context


# Your analyses / code below

In [3]:
import datetime

## Read in Shyam's input list

In [4]:
# Read the file into a list (on driver)
with open("openintel_tranco_scrubber_covered_ip_2024.txt", "r") as f:
    lines = f.readlines()

# Remove newline characters
lines = [line.strip() for line in lines]

# Convert list to DataFrame
tranco_df = spark.createDataFrame([(line,) for line in lines], ["ip4_address"])

tranco_df.persist()

# Show DataFrame
tranco_df.show(truncate=False)



+---------------+
|ip4_address    |
+---------------+
|170.114.52.2   |
|141.193.213.20 |
|141.193.213.21 |
|141.193.213.10 |
|141.193.213.11 |
|208.74.121.151 |
|208.74.123.84  |
|72.13.63.40    |
|209.131.162.45 |
|129.176.1.88   |
|31.222.67.112  |
|198.49.23.176  |
|198.49.23.177  |
|198.185.159.176|
|198.185.159.177|
|205.178.187.13 |
|217.114.94.2   |
|198.49.23.144  |
|198.49.23.145  |
|198.185.159.144|
+---------------+
only showing top 20 rows



## Load in OpenINTEL resources

In [5]:
base_path = 's3a://openintel/category=fdns/type=warehouse/'
print(f'Base path: {base_path}')

def compute(dataset_name: str, date_of_interest: datetime.date, small_df):
    print(f'Generating spark recipe for {dataset_name}')
    measurement_path = '{base_path}source={source}/year={year}/month={month:02d}/day={day:02d}/'.format(
        base_path=base_path,
        source=dataset_name,
        year=date_of_interest.year,
        month=date_of_interest.month,
        day=date_of_interest.day
    )
    print(f'Data path: {measurement_path}')
    
    # Load in data
    zone_df = spark.read.option('basePath', base_path).parquet(measurement_path).filter(
        # Only keep rows with an IP4 address
        psf.col('ip4_address').isNotNull()  
    ).select(
        # Select columns we're interested in
        'query_name',
        'ip4_address'
    ).distinct().persist()


    unique_domains_with_a_in_zone_df = zone_df.select(
        # We want the number of unique query_names that have at least one A record.
        'query_name'
    ).distinct(
        # Make sure the names are unique, as a query_name might have multiple A records
    ).withColumn(
        # Create a new column that is the zone (last label of query_name)
        'zone', 
        psf.element_at(
            psf.split(psf.col('query_name'), '\\.'),  # Split on '.'
            -2  # Second to last element, since the query_name includes the root dot.
        )
    ).groupBy("zone").agg(
        # Grouped by zone, get counts for query_names starting with www. and those without, and a total count.
        psf.count(psf.when(psf.col('query_name').startswith('www.'), True)).alias('www_count_in_zone'),
        psf.count(psf.when(~psf.col('query_name').startswith('www.'), True)).alias('non_www_count_in_zone'),
        psf.count(psf.col('query_name')).alias('total_count_in_zone')
    )

    unique_domains_with_a_in_zone_and_input_list_df = zone_df.join(
        # Join on the input dataset. 
        # This dataset is small, so we broadcast it to all compute nodes
        # Inner join; means that it includes all domains that have at least 1 A record with an IP in the input list
        psf.broadcast(tranco_df),
        on='ip4_address',
        how='inner'
    ).select(
        'query_name'
    ).distinct(
        # We're only interested in the unique number of domains.
    ).withColumn(
        # Create a new column that is the zone (last label of query_name)
        'zone', 
        psf.element_at(
            psf.split(psf.col('query_name'), '\\.'),  # Split on '.'
            -2  # Second to last element, since the query_name includes the root dot.
        )
    ).groupBy("zone").agg(
        # Grouped by zone, get counts for query_names starting with www. and those without, and a total.
        psf.count(psf.when(psf.col('query_name').startswith('www.'), True)).alias('www_count_in_list'),
        psf.count(psf.when(~psf.col('query_name').startswith('www.'), True)).alias('non_www_count_in_list'),
        psf.count(psf.col('query_name')).alias('total_count_in_list')
    )

    result_df = unique_domains_with_a_in_zone_and_input_list_df.join(
        unique_domains_with_a_in_zone_df,
        on='zone',
        how='outer'
    ).withColumn(
        'www_percentage',
        psf.round((psf.col('www_count_in_list') / psf.col('www_count_in_zone')) * 100, 2)
    ).withColumn(
        'non_www_percentage',
        psf.round((psf.col('non_www_count_in_list') / psf.col('non_www_count_in_zone')) * 100, 2)
    ).withColumn(
        'total_percentage',
        psf.round((psf.col('total_count_in_list') / psf.col('total_count_in_zone')) * 100, 2)
    ).select(
        # Select better ordering.
        'zone',
        
        'www_count_in_list',
        'www_count_in_zone',
        'www_percentage',

        'non_www_count_in_list',
        'non_www_count_in_zone',
        'non_www_percentage',

        'total_count_in_list',
        'total_count_in_zone',
        'total_percentage'
    )
    
    return result_df


Base path: s3a://openintel/category=fdns/type=warehouse/


In [6]:
doi = datetime.date(year=2024, month=1, day=1)
print(f'Date of interest: {doi}')

Date of interest: 2024-01-01


In [7]:
com_df = compute('com', doi, tranco_df).persist()
com_df.show()

Generating spark recipe for com
Data path: s3a://openintel/category=fdns/type=warehouse/source=com/year=2024/month=01/day=01/
+----+-----------------+-----------------+--------------+---------------------+---------------------+------------------+-------------------+-------------------+----------------+
|zone|www_count_in_list|www_count_in_zone|www_percentage|non_www_count_in_list|non_www_count_in_zone|non_www_percentage|total_count_in_list|total_count_in_zone|total_percentage|
+----+-----------------+-----------------+--------------+---------------------+---------------------+------------------+-------------------+-------------------+----------------+
| com|          5994267|        133042892|          4.51|              7804309|            134581564|               5.8|           13798576|          267624456|            5.16|
+----+-----------------+-----------------+--------------+---------------------+---------------------+------------------+-------------------+-------------------+--

In [8]:
org_df = compute('org', doi, tranco_df).persist()
org_df.show()

Generating spark recipe for org
Data path: s3a://openintel/category=fdns/type=warehouse/source=org/year=2024/month=01/day=01/
+----+-----------------+-----------------+--------------+---------------------+---------------------+------------------+-------------------+-------------------+----------------+
|zone|www_count_in_list|www_count_in_zone|www_percentage|non_www_count_in_list|non_www_count_in_zone|non_www_percentage|total_count_in_list|total_count_in_zone|total_percentage|
+----+-----------------+-----------------+--------------+---------------------+---------------------+------------------+-------------------+-------------------+----------------+
| org|           477996|          9252031|          5.17|               636956|              9366865|               6.8|            1114952|           18618896|            5.99|
+----+-----------------+-----------------+--------------+---------------------+---------------------+------------------+-------------------+-------------------+--

In [9]:
net_df = compute('net', doi, tranco_df).persist()
net_df.show()

Generating spark recipe for net
Data path: s3a://openintel/category=fdns/type=warehouse/source=net/year=2024/month=01/day=01/
+----+-----------------+-----------------+--------------+---------------------+---------------------+------------------+-------------------+-------------------+----------------+
|zone|www_count_in_list|www_count_in_zone|www_percentage|non_www_count_in_list|non_www_count_in_zone|non_www_percentage|total_count_in_list|total_count_in_zone|total_percentage|
+----+-----------------+-----------------+--------------+---------------------+---------------------+------------------+-------------------+-------------------+----------------+
| net|           394141|         10241315|          3.85|               559276|             10391510|              5.38|             953417|           20632825|            4.62|
+----+-----------------+-----------------+--------------+---------------------+---------------------+------------------+-------------------+-------------------+--

In [10]:
closedcc_df = compute('closedcc', doi, tranco_df).persist()
closedcc_df.show()

Generating spark recipe for closedcc
Data path: s3a://openintel/category=fdns/type=warehouse/source=closedcc/year=2024/month=01/day=01/
+--------+-----------------+-----------------+--------------+---------------------+---------------------+------------------+-------------------+-------------------+----------------+
|    zone|www_count_in_list|www_count_in_zone|www_percentage|non_www_count_in_list|non_www_count_in_zone|non_www_percentage|total_count_in_list|total_count_in_zone|total_percentage|
+--------+-----------------+-----------------+--------------+---------------------+---------------------+------------------+-------------------+-------------------+----------------+
|      us|            41934|          1586024|          2.64|                73223|              1618513|              4.52|             115157|            3204537|            3.59|
|      co|           104473|          2735573|          3.82|               184135|              2807429|              6.56|            

In [11]:
combined_df = com_df.union(org_df).union(net_df).union(closedcc_df)

combined_df.orderBy(
    psf.col('total_count_in_zone').desc()
).show()

+--------+-----------------+-----------------+--------------+---------------------+---------------------+------------------+-------------------+-------------------+----------------+
|    zone|www_count_in_list|www_count_in_zone|www_percentage|non_www_count_in_list|non_www_count_in_zone|non_www_percentage|total_count_in_list|total_count_in_zone|total_percentage|
+--------+-----------------+-----------------+--------------+---------------------+---------------------+------------------+-------------------+-------------------+----------------+
|     com|          5994267|        133042892|          4.51|              7804309|            134581564|               5.8|           13798576|          267624456|            5.16|
|     net|           394141|         10241315|          3.85|               559276|             10391510|              5.38|             953417|           20632825|            4.62|
|     org|           477996|          9252031|          5.17|               636956|       

In [12]:
oi_clearcache()

In [13]:
oi_stop()

Cleared cache and stopped spark context
